# Test the L0 processor v1

In [ ]:
# For testing, don't commit
import sys
sys.path.insert(0, "/home/jgaucher/projects/rspy/github/rs-demo/notebooks")
import resources.test_localhost

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

In [ ]:
# Other imports
import json
import os.path as osp

from rs_common.prefect_utils import *

## Read the tasktables

Documentation: https://cpm.pages.eopf.copernicus.eu/eopf-cpm/main/processor-orchestration-guide/tasktables.html#tasktables

<div class="alert alert-block alert-warning">

Note: for now the L0 processor returns dummy values that are not usable.
</div>

In [7]:
for process_id in "s1_l0", "s3_l0":
    tasktable: dict = dpr_client.get_process(process_id)
    print(f"Tasktable for {process_id!r}:\n{json.dumps(tasktable, indent=2)}\n\n")

Tasktable for 's1_l0':
{
  "version": "TODO_version",
  "baseline_collection": "TODO_baseline_collection",
  "processor_name": "S1L0Processor",
  "input_products": [
    {
      "name": "cadus",
      "mode": "always",
      "mandatory": true,
      "alternatives": [
        {
          "order": "1",
          "stac_query": {
            "filter-lang": "cql2-json",
            "filter": {
              "op": "and",
              "args": [
                {
                  "op": "=",
                  "args": [
                    {
                      "property": "product_type"
                    },
                    "global_variables.product_type"
                  ]
                },
                {
                  "op": "t_intersects",
                  "args": [
                    {
                      "interval": [
                        {
                          "property": "start_datetime"
                        },
                        {
                   

<div class="alert alert-block alert-warning">

To be discussed: should the configuration and payload files for the processors be available from:

  * rs-dpr-service ?
  * rs-client-libraries ?
  * The user local disk ? (like in this demo)
</div>

## Run the L0 processor

In [8]:
# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config_dir = osp.join(s3_base, "config")
s3_output_dir = osp.join(s3_base, "output")
s3_report_dir = osp.join(s3_base, "reports")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./config", s3_config_dir)

# Data properties
s1_short = {
    "payload_subpath": "s1/iw_joborder.short.yaml",
    "s3_output_dir": f"{s3_output_dir}/s1.short",
    "s3_report_dir": f"{s3_report_dir}/s1.short",
}
s1 = {
    "payload_subpath": "s1/iw_joborder.yaml",
    "s3_output_dir": f"{s3_output_dir}/s1",
    "s3_report_dir": f"{s3_report_dir}/s1",
}
s3 = {
    "payload_subpath": "s3/s3_dordop_payload.yaml",
    "s3_output_dir": f"{s3_output_dir}/s3",
    "s3_report_dir": f"{s3_report_dir}/s3",
}

# Update local configuration files depending on the environment, 
# and upload them again to the s3 bucket.

# Update secret file
await dpr_client.update_configuration(
    local_path = "./config/secrets.json",
    s3_path = osp.join(s3_config_dir, "secrets.json"),
)

# Update payload files
for data in s1_short, s1, s3:
    await dpr_client.update_configuration(
        local_path = osp.join("./config", data["payload_subpath"]),
        s3_path = osp.join(s3_config_dir, data["payload_subpath"]),
        is_payload = True,
        # Specific environment variables to expand in the payload file
        PREFECT_BUCKET_NAME=os.environ["PREFECT_BUCKET_NAME"], 
        OUTPUT_DIR=data["s3_output_dir"],
    )

17:42:57.446 | INFO    | prefect.S3Bucket - Uploading from 'config/secrets.json' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/secrets.json'.

17:42:57.448 | INFO    | prefect.S3Bucket - Uploading from 'config/l0_dask_configuration.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/l0_dask_configuration.yaml'.

17:42:57.449 | INFO    | prefect.S3Bucket - Uploading from 'config/logging_config.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/logging_config.yaml'.

17:42:57.452 | INFO    | prefect.S3Bucket - Uploading from 'config/s1/iw_joborder.short.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_joborder.short.yaml'.

17:42:57.454 | INFO    | prefect.S3Bucket - Uploading from 'config/s1/iw_configuration.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_configuration.yaml'.

...

17:42:57.542 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpzey_9bmo' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_joborder.short.yaml'.

17:42:57.557 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmp7ei30bf4' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/output/s1/S1A_20240410083700053369/.empty'.

17:42:57.571 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmp5a3sasw9' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_joborder.yaml'.

17:42:57.586 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmp1lhs8dhd' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/output/s3/S3A_20250612034536048530/.empty'.

17:42:57.597 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpyuiyr__r' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'.

### S1 short data

In [ ]:
s3_output_dir = s1_short["s3_output_dir"]
print(f"Remove existing zarr products from: {s3_output_dir!r}")
s3_delete(s3_output_dir)

# Run processor
dpr_client.run_process(
    "s1_l0",
    {
        "s3_config_dir": s3_config_dir,
        "payload_subpath": s1_short["payload_subpath"],
        "s3_report_dir": s1_short["s3_report_dir"],
    }
)

Remove existing zarr products from: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short'


In [ ]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

In [ ]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)